# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [48]:
# load enviornment

%load_ext dotenv
%dotenv

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [49]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [50]:
import os
from glob import glob

# load PRICE_DATA enviornment variable
PRICE_DATA = os.getenv("PRICE_DATA")

# locate all .parquet files using glob (recursively)
parquet_files = glob(os.path.join(PRICE_DATA, '**', '*.parquet'), recursive=True)

# read parquet files and set index as 'ticker'
dd_px = dd.read_parquet(parquet_files).set_index("ticker")

# sample output
print(dd_px.head())

             Date   Open   High    Low  Close  Adj Close      Volume   source  \
ticker                                                                          
ACN    2001-07-19  15.10  15.29  15.00  15.17  11.404394  34994300.0  ACN.csv   
ACN    2001-07-20  15.05  15.05  14.80  15.01  11.284108   9238500.0  ACN.csv   
ACN    2001-07-23  15.00  15.01  14.55  15.00  11.276587   7501000.0  ACN.csv   
ACN    2001-07-24  14.95  14.97  14.70  14.86  11.171341   3537300.0  ACN.csv   
ACN    2001-07-25  14.70  14.95  14.65  14.95  11.238999   4208100.0  ACN.csv   

        Year  
ticker        
ACN     2001  
ACN     2001  
ACN     2001  
ACN     2001  
ACN     2001  


For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [51]:

# create feature for close lag

dd_feat = dd_px.groupby('ticker', group_keys=False).apply(
    lambda x: x.assign(
        Close_lag_1 = x['Close'].shift(1),
        Adj_Close_lag_1 = x['Adj Close'].shift(1)
    ),

    # create feature for adj close lag

    meta=dd_px.assign(
        Close_lag_1 = 0.0,
        Adj_Close_lag_1 = 0.0
    )
)

# create feature for returns based on values of close and close lag

dd_feat = dd_feat.assign(
    Returns = lambda x: x['Close'] / x['Close_lag_1'] - 1,

    # create feature for high-low range

    High_Low_Range = lambda x: x['High'] - x['Low']
)

# sample output

print(dd_feat.head())

             Date   Open   High    Low  Close  Adj Close      Volume   source  \
ticker                                                                          
ACN    2001-07-19  15.10  15.29  15.00  15.17  11.404394  34994300.0  ACN.csv   
ACN    2001-07-20  15.05  15.05  14.80  15.01  11.284108   9238500.0  ACN.csv   
ACN    2001-07-23  15.00  15.01  14.55  15.00  11.276587   7501000.0  ACN.csv   
ACN    2001-07-24  14.95  14.97  14.70  14.86  11.171341   3537300.0  ACN.csv   
ACN    2001-07-25  14.70  14.95  14.65  14.95  11.238999   4208100.0  ACN.csv   

        Year  Close_lag_1  Adj_Close_lag_1   Returns  High_Low_Range  
ticker                                                                
ACN     2001          NaN              NaN       NaN            0.29  
ACN     2001        15.17        11.404394 -0.010547            0.25  
ACN     2001        15.01        11.284108 -0.000666            0.46  
ACN     2001        15.00        11.276587 -0.009333            0.27  
ACN   

+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [52]:
# convert dask df to pandas df

pdf_feat = dd_feat.compute()

# add new feature column for 10 day rolling average

pdf_feat['10_day_roll_avg'] = (
    pdf_feat
    .groupby('ticker')['Returns']
    .transform(lambda x: x.rolling(10, min_periods=1).mean())
)

# sample output

print(pdf_feat.head())

             Date        Open        High         Low       Close   Adj Close  \
ticker                                                                          
ACN    2018-01-02  153.500000  154.100006  152.779999  153.839996  148.655960   
ACN    2018-01-03  152.990005  154.990005  152.990005  154.550003  149.342056   
ACN    2018-01-04  155.000000  156.860001  154.770004  156.380005  151.110382   
ACN    2018-01-05  156.610001  157.720001  156.130005  157.669998  152.356918   
ACN    2018-01-08  157.369995  159.009995  156.839996  158.929993  153.574463   

           Volume   source  Year  Close_lag_1  Adj_Close_lag_1   Returns  \
ticker                                                                     
ACN     3061900.0  ACN.csv  2018          NaN              NaN       NaN   
ACN     2064200.0  ACN.csv  2018   153.839996       148.655960  0.004615   
ACN     1777000.0  ACN.csv  2018   154.550003       149.342056  0.011841   
ACN     1597600.0  ACN.csv  2018   156.380005       

Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

No it was not necessary to calculate the 10 day rolling average in dask, it can be calculated in both pandas and dask. Dask is just more efficent with larger data sets. However because Dask breaks up the data into chunks and calculates then in parallel, this could cause issues with a rolling data calculation. Although apparently Dask has something built in to accomodate this.

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.